In [1]:
from pyspark.sql import SparkSession

In [2]:
#Must set this env variable to avoid warnigns
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = '1'

In [3]:
import pandas as pd
import pyspark.pandas as ps

In [4]:
# Initialize Spark Session
spark = SparkSession.builder \
.appName("Pandas Integration With Spark") \
.config("spark.sql.ansi.enabled", "false") \
.config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1") \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-pattern-layout-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/05 16:59:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
#1. Create a pandas dataframe
pandas_df = pd.DataFrame({
    "id": [1,2,3,4,5],
    "name": ["Alice", "Bob", "Charlie", "David", "Emma"],
    "age": [25,30,35,40,45]
})

In [6]:
#2. Convert pandas dataframe to spark dataframe

spark_df = spark.createDataFrame(pandas_df)

In [7]:
print("\nSchema of Spark DataFrame:")
spark_df.printSchema()


Schema of Spark DataFrame:
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [8]:
print("\nSpark DataFrame:")
spark_df.show()


Spark DataFrame:


+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 35|
|  4|  David| 40|
|  5|   Emma| 45|
+---+-------+---+



In [9]:
# 3. Perform transformations on Spark DataFrame
filtered_spark_df = spark_df.filter(spark_df.age > 30)
print("\n Filtered Spark DataFrame (age > 30):")
filtered_spark_df.show()


 Filtered Spark DataFrame (age > 30):
+---+-------+---+
| id|   name|age|
+---+-------+---+
|  3|Charlie| 35|
|  4|  David| 40|
|  5|   Emma| 45|
+---+-------+---+



In [10]:
#4. Convert Spark DataFrame back to Pandas DataFrame

converted_pandas_df = filtered_spark_df.toPandas()

In [11]:
print("Converted Pandas DataFrame:")
print(converted_pandas_df)

Converted Pandas DataFrame:
   id     name  age
0   3  Charlie   35
1   4    David   40
2   5     Emma   45


In [12]:
# 5. Use pandas-on-Spark for scalable Pandas operations
ps_df = ps.DataFrame(pandas_df)

In [13]:
# Perform a pandas-like operation in Spark
print("\n Using a pandas-on-Spark (Incrementing age by 1):")
ps_df["age"] = ps_df["age"] + 1
print(ps_df)


 Using a pandas-on-Spark (Incrementing age by 1):
   id     name  age
0   1    Alice   26
1   2      Bob   31
2   3  Charlie   36
3   4    David   41
4   5     Emma   46


In [14]:
# Convert pandas-on-Spark DataFrame to Spark Dataframe
converted_spark_df = ps_df.to_spark()
print("\n Converted Spark DataFrame from pandas-on-Spark:")
converted_spark_df.show()

/opt/spark/python/pyspark/pandas/utils.py:1017: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)



 Converted Spark DataFrame from pandas-on-Spark:
+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 26|
|  2|    Bob| 31|
|  3|Charlie| 36|
|  4|  David| 41|
|  5|   Emma| 46|
+---+-------+---+



In [15]:
spark.stop()